# Niva OpenEnv Training Notebook

## Hackathon-Aligned LLM RL Pipeline

This notebook uses the refactored MAAS backend files directly:
- `environment.py` for the OpenEnv-style maternal-health environment
- `xai_reward_model.py` for deterministic reward scoring
- `train_trl.py` for PPO training with Hugging Face TRL

It replaces the old inline Q-learning notebook with a prompt-based LLM training flow.

## Cell 1 - Install Dependencies

Run this in Colab after uploading or cloning the `MAAS` folder into `/content/MAAS`.

In [ ]:
%pip install openenv-core trl transformers datasets peft accelerate sqlalchemy pydantic fastapi uvicorn torch -q
print('Dependencies installed')

## Cell 2 - Point Python to the MAAS Repo

In [ ]:
import os
import sys

REPO_ROOT = '/content/MAAS'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Using repo root:', REPO_ROOT)
print('Files:', sorted(os.listdir(REPO_ROOT))[:15])

## Cell 3 - Import the Refactored Environment and Reward Model

In [ ]:
from environment import PrenatalEnvironment, parse_llm_output
from train_trl import build_prompt, run_training
from xai_reward_model import calculate_reward

env = PrenatalEnvironment()
print('Environment loaded:', env.env_name, env.env_version)
print('OpenEnv available:', env.openenv_available)

## Cell 4 - Inspect One LLM-Ready Observation

In [ ]:
USER_ID = 1
prompt_obs = env.reset(USER_ID)

print('Structured observation:')
print(prompt_obs.observation.model_dump())
print('\nLLM text observation:\n')
print(prompt_obs.text_observation)
print('\nPrompt preview:\n')
print(build_prompt(prompt_obs.text_observation))

## Cell 5 - Score a Sample LLM Response

In [ ]:
sample_response = '''{
  "condition": "preeclampsia",
  "urgency": "go_to_hospital_today",
  "rationale": "Danger blood-pressure and symptom flags need urgent escalation."
}'''

action = parse_llm_output(sample_response)
breakdown = calculate_reward(action.target, action.urgency, prompt_obs.observation)

print('Parsed action:', action.model_dump())
print('Reward:', breakdown.reward)
print('Reference condition:', breakdown.reference_condition)
print('Reference urgency:', breakdown.reference_urgency)
print('Under-escalated:', breakdown.under_escalated)
print('Rationale:', breakdown.rationale)

## Cell 6 - Run Minimal PPO Training

In [ ]:
from argparse import Namespace

args = Namespace(
    model_name='meta-llama/Meta-Llama-3-8B-Instruct',
    output_dir='./artifacts/niva-trl-ppo',
    user_ids='1',
    epochs=1,
    batch_size=1,
    mini_batch_size=1,
    learning_rate=1e-5,
    max_new_tokens=160,
    load_in_4bit=False,
)

print(args)

## Cell 7 - Start Training

Uncomment the last line when the model, database, and repo files are available in Colab.

In [ ]:
# run_training(args)
print('Training cell is ready. Remove the comment to launch PPO training.')

## Cell 8 - Submission Notes

For the hackathon demo, capture these outputs:
1. One `text_observation` example.
2. One model JSON action.
3. One reward breakdown from `calculate_reward()`.
4. PPO training logs from `run_training(args)`.

That evidence is usually much stronger than screenshots of an old linear-Q baseline.